# Step-level selection, Stage 0: is there anything to choose between?

Evaluation only, no training, on the SIM-CoT GPT-2 checkpoint and the 500 validation questions. Plan and decision rule:
`docs/brainstorm/STEP_SELECTION.md`.

For every question and each of two sampling seeds, `scripts/step_select/eval.py` runs:
- **Greedy:** the model as released. It must reproduce Stage 0's own answers on 500/500 questions.
- **Chains:** 18 independent runs with MC dropout (p = 0.2) at every latent step.
- **Branches:** greedy everywhere except one step, which is rerun 3 times with dropout and then finished normally.
  That gives 18 branches, each departing from greedy at a single step.
- **An untrained selector:** at each step, the model's own step plus 3 dropout reruns. SIM-CoT's step decoder reads all 4,
  and a fixed rule keeps one.

The report cell gives the identity check, how much room each kind of selection has, whether the decoder rule can tell
right candidates from wrong ones, and the preregistered verdict.

Each finished seed is saved to Drive, so **rerunning after a disconnect skips it**.

**Before running:** Runtime → Change runtime type → **GPU**; a T4 is enough. Then Runtime → **Run all**, and keep the tab
open until the last cell downloads the results.

In [ ]:
# A run is identified by these settings; rerunning with the same values resumes it.
SEEDS = [1, 2]            # sampling seeds; the decision pools both
P = 0.2                   # MC-dropout rate, as in LatentTTS
CHAINS = 18               # independent chains: with greedy, 19 finished answers, as many as greedy plus the branches
CANDIDATES = 4            # per step, the model's own step included
EVAL_LIMIT = None         # e.g. 50 for a quick look; None scores all 500 validation questions
RUN_TAG = ""              # e.g. "-try2" to start a run over under a new name
REPO_REF = "main"         # the branch or commit holding scripts/step_select

In [ ]:
from pathlib import Path
import os, runpy, shutil, subprocess, sys
from google.colab import drive

drive.mount("/content/drive")
DRIVE = Path("/content/drive/MyDrive/dynamic-state-anchoring")
RUN_NAME = f"step-select-stage0-p{P}-c{CANDIDATES}-n{CHAINS}{RUN_TAG}"
BACKUP_DIR = DRIVE / "step-select" / RUN_NAME
gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                     capture_output=True, text=True)
if gpu.returncode:
    raise SystemExit("No GPU: Runtime > Change runtime type > GPU, then Runtime > Run all.")
print("GPU:", gpu.stdout.strip())

PINNED = BACKUP_DIR / "commit.txt"  # a started run keeps the code it started with
ref = PINNED.read_text().strip() if PINNED.exists() else REPO_REF
REPO_DIR = Path("/content/dynamic-state-anchoring")
if not REPO_DIR.exists():
    subprocess.run(["git", "clone", "https://github.com/sttruji/dynamic-state-anchoring.git", str(REPO_DIR)],
                   check=True)
subprocess.run(["git", "fetch", "origin", ref], cwd=REPO_DIR, check=True)
subprocess.run(["git", "checkout", "--detach", "FETCH_HEAD"], cwd=REPO_DIR, check=True)
os.chdir(REPO_DIR)
COMMIT = subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip()
assert Path("scripts/step_select/eval.py").exists(), f"{ref} has no scripts/step_select; push it first"
BACKUP_DIR.mkdir(parents=True, exist_ok=True)
PINNED.write_text(COMMIT)
print("Code:", COMMIT, f"(from {ref})")

In [ ]:
VENV = Path("/content/dsa-venv")
subprocess.run([sys.executable, "scripts/setup_colab.py", "--venv-dir", str(VENV),
                "--requirements", "requirements-colab.txt"], check=True)
PYTHON = str(VENV / "bin/python")
os.environ["DSA_DATA_DIR"] = str(DRIVE / "data")
os.environ["HF_HOME"] = "/content/hf-cache"  # local disk: the school Drive is nearly full
os.environ["TOKENIZERS_PARALLELISM"] = "false"
WORK = Path("/content/step-select")  # fast local disk; finished work is copied to BACKUP_DIR
WORK.mkdir(exist_ok=True)
run_logged = runpy.run_path("scripts/colab_process.py")["run_logged"]

def run(*args):
    run_logged([PYTHON, *map(str, args)], cwd=REPO_DIR, log_dir=BACKUP_DIR / "logs")

run("scripts/setup_data.py", "--eval-only")
CHECKPOINT = DRIVE / "checkpoints/simcot-gpt2"
run("scripts/latent_gate/fetch_simcot.py", "--output", CHECKPOINT)
EVALS = BACKUP_DIR / "evals"
limit = [] if EVAL_LIMIT is None else ["--limit", EVAL_LIMIT]

In [ ]:
for seed in SEEDS:
    target = EVALS / f"seed{seed}.jsonl"
    if target.exists() and EVAL_LIMIT is None:
        print(f"seed {seed}: already on Drive")
        continue
    local = WORK / f"seed{seed}.jsonl"
    run("scripts/step_select/eval.py", "--checkpoint", CHECKPOINT, "--dataset", "gsm8k_valid", "--output", local,
        "--p", P, "--seed", seed, "--chains", CHAINS, "--candidates", CANDIDATES, "--device", "cuda", *limit)
    target.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(local, target)

In [ ]:
# The reference is Stage 0 of the inner loop: the checkpoint's own answers on the same 500 questions.
run("scripts/step_select/report.py", *[EVALS / f"seed{seed}.jsonl" for seed in SEEDS],
    "--reference", "results/inner_loop/stage0/evals/base-simcot/gsm8k_valid.jsonl")

In [ ]:
# The evaluations, for the repo's results folder.
from google.colab import files

bundle = Path(f"/content/{RUN_NAME}")
shutil.rmtree(bundle, ignore_errors=True)
shutil.copytree(EVALS, bundle / "evals")
(bundle / "commit.txt").write_text(COMMIT)
files.download(shutil.make_archive(str(bundle), "zip", bundle))
print(f"Downloading {bundle}.zip. If it does not start, get it from the Files panel on the left.")

**Reading the result**
- **Identity** must say PASS: with dropout off, the code is the released model. If it says FAIL, stop and report it.
- **Choosing among finished chains** and **choosing per step** say how much each kind of selection could gain at best:
  how many more questions have at least one right answer among the candidates than greedy gets right.
- **Per step** shows where the branches rescue answers, and where they break answers greedy had right.
- **Signal** covers the (question, step) sets whose candidates mix right and wrong continuations. It asks whether the
  fixed decoder rule picks a right one more often than chance, and gives each feature's AUC, the chance it ranks a right
  candidate above a wrong one.
- **The verdict:**
  - **GO** needs the branches to cover at least 8% more questions than greedy (40 of 500) and a decoder signal at
    p < 0.05.
  - **STOP** if the branches cover less than 4% more, or if there is no signal and the untrained selector does no better
    than greedy.
  - **AMBIGUOUS** otherwise.

**Sharing:** the last cell downloads a zip of the evaluations. Point Claude at it.